# Blackbird (MIT) + Spyx New Architectures

This notebook targets the Blackbird UAV dataset.

Status in Tonic:
- No direct Blackbird loader in `tonic.datasets` in this environment.

Goal:
- Use local Blackbird exports if available.
- Compare Spyx architectures under fast motion dynamics.

In [1]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import spyx.models as fm

DATA_ROOT = Path("../data/Blackbird")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_blackbird():
    import tonic.datasets as d
    return any("blackbird" in n.lower() for n in dir(d))

print("tonic_has_blackbird =", tonic_has_blackbird())
print("expected local artifacts: imu.csv, trajectory.csv, events.* or frames/")

/Users/vincent/Work/autoresearch-mlx/.bench-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_ROOT: /Users/vincent/Work/autoresearch-mlx/spyx/research/data/Blackbird
tonic_has_blackbird = False
expected local artifacts: imu.csv, trajectory.csv, events.* or frames/


In [2]:
def make_blackbird_target(batch=4, dim=6):
    return jnp.zeros((batch, dim), dtype=jnp.float32)


def synthetic_blackbird_batch(batch=4, T=20, H=64, W=64, C=2, seed=1):
    rng = np.random.default_rng(seed)
    x = rng.poisson(0.03, size=(T, batch, H, W, C)).astype(np.float32)
    return jnp.asarray(x), make_blackbird_target(batch, dim=6)


def blackbird_model_bank(input_hw=(64, 64), input_channels=2, output_dim=6):
    conv_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=32, channels2=64, output_dim=output_dim)
    sparse_cfg = fm.SparseConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=32, channels2=64, output_dim=output_dim, event_threshold=0.0)
    return {
        "conv": lambda x: fm.ConvLIFSNN(conv_cfg)(x),
        "ternary": lambda x: fm.TernaryConvLIFSNN(conv_cfg)(x),
        "sparse": lambda x: fm.SparseEventConvLIFSNN(sparse_cfg)(x),
    }


x, y = synthetic_blackbird_batch()
models = blackbird_model_bank()

for name, fn in models.items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x)
    pred, aux = net.apply(params, x)
    mse = jnp.mean((pred - y) ** 2)
    print(name, "pred", pred.shape, "mse", float(mse), "spike_rate", np.asarray(aux["spike_rate"]))

conv pred (4, 6) mse 0.0 spike_rate [0.0013876 0.       ]
ternary pred (4, 6) mse 0.0 spike_rate [0. 0.]
sparse pred (4, 6) mse 0.0 spike_rate [0.0013876 0.       ]


## Optuna Sweep

Blackbird emphasizes aggressive motion, so this search focuses on sparse and ternary event encoders under a shared regression target.

In [3]:
import optuna


def blackbird_objective(trial):
    arch = trial.suggest_categorical("arch", ["conv", "ternary", "sparse"])
    channels1 = trial.suggest_categorical("channels1", [24, 32, 40])
    channels2 = trial.suggest_categorical("channels2", [48, 64, 80])
    beta = trial.suggest_float("beta", 0.8, 0.98)
    threshold = trial.suggest_float("threshold", 0.7, 1.5)

    x_local, y_local = synthetic_blackbird_batch(batch=4, T=20, H=64, W=64, C=2, seed=trial.number)

    if arch == "sparse":
        cfg = fm.SparseConvConfig(
            input_hw=(64, 64), input_channels=2,
            channels1=channels1, channels2=channels2,
            output_dim=6, beta=beta, threshold=threshold, event_threshold=0.0,
        )
        forward = lambda xb: fm.SparseEventConvLIFSNN(cfg)(xb)
    else:
        cfg = fm.ConvConfig(
            input_hw=(64, 64), input_channels=2,
            channels1=channels1, channels2=channels2,
            output_dim=6, beta=beta, threshold=threshold,
        )
        if arch == "ternary":
            forward = lambda xb: fm.TernaryConvLIFSNN(cfg)(xb)
        else:
            forward = lambda xb: fm.ConvLIFSNN(cfg)(xb)

    net = hk.without_apply_rng(hk.transform(forward))
    params = net.init(jax.random.PRNGKey(0), x_local)
    pred, aux = net.apply(params, x_local)
    mse = jnp.mean((pred - y_local) ** 2)
    sparsity_bonus = 0.01 * jnp.mean(jnp.asarray(aux["spike_rate"]))
    return float(mse + sparsity_bonus)


study = optuna.create_study(direction="minimize", study_name="blackbird_spyx_arch_search")
study.optimize(blackbird_objective, n_trials=8)
study.best_trial.params, study.best_value

[I 2026-03-24 08:01:50,666] A new study created in memory with name: blackbird_spyx_arch_search
[I 2026-03-24 08:01:52,394] Trial 0 finished with value: 0.0 and parameters: {'arch': 'ternary', 'channels1': 40, 'channels2': 48, 'beta': 0.934300076649827, 'threshold': 1.105255611687618}. Best is trial 0 with value: 0.0.
[I 2026-03-24 08:01:53,167] Trial 1 finished with value: 0.0 and parameters: {'arch': 'ternary', 'channels1': 32, 'channels2': 80, 'beta': 0.8662852742082672, 'threshold': 1.1949236222840949}. Best is trial 0 with value: 0.0.
[I 2026-03-24 08:01:53,338] Trial 2 finished with value: 0.0 and parameters: {'arch': 'ternary', 'channels1': 32, 'channels2': 80, 'beta': 0.9278327199392225, 'threshold': 0.8257228849749414}. Best is trial 0 with value: 0.0.
[I 2026-03-24 08:01:54,203] Trial 3 finished with value: 1.835474176914431e-05 and parameters: {'arch': 'sparse', 'channels1': 24, 'channels2': 64, 'beta': 0.9141373398032022, 'threshold': 0.8505720587974812}. Best is trial 0 wi

({'arch': 'ternary',
  'channels1': 40,
  'channels2': 48,
  'beta': 0.934300076649827,
  'threshold': 1.105255611687618},
 0.0)

## Next Steps for Real Blackbird Runs

1. Export synchronized event/image and IMU streams to `../data/Blackbird`.
2. Build a sequence index with train/val/test by flight condition.
3. Extend the notebook with trajectory loss terms and multi-sequence evaluation.